# RAGAS Metrics Exploration

This notebook demonstrates how to use individual RAGAS metrics for evaluating RAG systems.

## Metrics Covered:
1. **Context Precision** - Measures the precision of retrieved contexts for answering the question
2. **Context Recall** - Measures how well the retrieved contexts cover the reference answer
3. **Faithfulness** - Measures how factually consistent a response is with the retrieved context
4. **Answer Relevancy** - Evaluates how relevant the answer is to the question
5. **Noise Sensitivity** - Measures how often the system makes errors with relevant/irrelevant contexts


In [1]:
!pip install ragas==0.4.2

  Using cached ragas-0.4.2-py3-none-any.whl.metadata (22 kB)
  Using cached nest_asyncio-1.6.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
  Using cached scikit_network-0.33.5-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.5 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached multiprocess-0.70.19-py312-none-any.whl.metadata (7.5 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached jiter-0.14.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.2 kB)
  Using cached rich-14.3.4-py3-none-any.whl.metadata (18 kB)
Using cached ragas-0.4.2-py3-none-any.whl (457 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 3.9 MB/s eta 0:00:00m eta 0:00:010:00:01
Using cached diskcache-5.6.3-py3-none-any.whl (45 kB)
Using cached appdirs-1.4.4-py2.py3-none-any.whl (9.6 k

In [2]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 1.9 MB/s eta 0:00:00m eta 0:00:01:01


In [ ]:
# Install required packages (if not already installed)
#pip install ragas==0.4.2 or 
#pip install -r requirements.txt

In [1]:
# Import required libraries
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings import HuggingFaceEmbeddings
from ragas.metrics.collections import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall,
    NoiseSensitivity,
)
import os
from dotenv import load_dotenv

load_dotenv()

# Setup OpenAI client
client = AsyncOpenAI(api_key = os.getenv("GROQ_API_KEY"), base_url = "https://api.groq.com/openai/v1" )

# Setup LLM for evaluation
llm = llm_factory("openai/gpt-oss-120b", client=client)

# Setup embeddings for AnswerRelevancy metric
embeddings = HuggingFaceEmbeddings(model = "all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 1. Context Recall Metric

**Purpose**: Evaluates context recall by classifying if statements in a reference answer can be attributed to the retrieved context. Uses an LLM to verify attribution for each statement and calculates recall as the proportion of attributed statements.

**Parameters**:
- `user_input`: The original question
- `retrieved_contexts`: List of retrieved context strings
- `reference`: The reference/gold standard answer

In [2]:
# Example 4: Context Recall Metric
scorer = ContextRecall(llm=llm)

# Sample data
user_input = "What is the capital of France and what is the lagest tower there?"

reference = "Paris is the capital of france. and Effile tower is the largest one"
retrieved_contexts = [
    "Paris is located in the north-central part of the country.",
    "capital is paris"
]
# Evaluate
result = await scorer.ascore(
    user_input=user_input,
    retrieved_contexts=retrieved_contexts,
    reference=reference
)

print(f"Context Recall Score: {result.value}")
print(f"Score Range: 0.0 to 1.0 (higher is better)")
print(f"Measures: How well the retrieved contexts cover the information in the reference answer")

Context Recall Score: 0.5
Score Range: 0.0 to 1.0 (higher is better)
Measures: How well the retrieved contexts cover the information in the reference answer


## 2 Context Precision Metric

**Purpose**: Evaluates whether retrieved contexts are useful for answering a question by comparing each context against a reference answer. The metric calculates average precision based on usefulness verdicts.

**Parameters**:
- `user_input`: The original question
- `reference`: The reference/gold standard answer
- `retrieved_contexts`: List of retrieved context strings

In [11]:
scorer = ContextPrecision(llm=llm)

# Sample data
user_input = "What is the capital of France?"
reference = "Paris is the capital of France."
retrieved_contexts = [
    
    "Berlin is the capital of Germany.",
    "France is a country in Western Europe ",
    "Paris is the capital and largest city of France, located in the north-central part of the country."
]

# Evaluate"
result = await scorer.ascore(
    user_input=user_input,
    reference=reference,
    retrieved_contexts=retrieved_contexts
)

print(f"Context Precision Score: {result.value}")
print(f"Score Range: 0.0 to 1.0 (higher is better)")


Context Precision Score: 0.3333333333
Score Range: 0.0 to 1.0 (higher is better)


## 3. Faithfulness Metric

**Purpose**: Measures how factually consistent a response is with the retrieved context. A response is considered faithful if all its claims can be supported by the context.

**Parameters**:
- `user_input`: The original question
- `response`: The generated response to evaluate
- `retrieved_contexts`: List of retrieved context strings


In [16]:
scorer = Faithfulness(llm=llm)

# Sample data
user_input = "Tell me about GPT-4?"
response = '''GPT-4 was released in March 2023.
It is a multimodal language model that understands both text and images.
It outperforms GPT-3.5 on many academic and professional benchmarks.
GPT-4 was initially available through ChatGPT Plus and the OpenAI API.
Some versions support a context window of up to 32,000 tokens.7'''
retrieved_contexts = [
    'OpenAI released GPT-4 in March 2023.',
    'GPT-4 is a multimodal large language model capable of understanding both text and images.',
    'The model supports a context window of up to 32,000 tokens in some versions.',
    'It performs better than GPT-3.5 on many professional and academic benchmarks.',
    'GPT-4 was initially made available through ChatGPT Plus and the OpenAI API.'
]

# Evaluate
result = await scorer.ascore(
    user_input=user_input,
    response=response,
    retrieved_contexts=retrieved_contexts
)

print(f"Faithfulness Score: {result.value}")
print(f"Score Range: 0.0 to 1.0 (higher is better)")


Faithfulness Score: 1.0
Score Range: 0.0 to 1.0 (higher is better)


## 4. Answer Relevancy Metric

**Purpose**: Evaluates how relevant the answer is to the question by generating multiple questions from the response and comparing them to the original question using semantic similarity.

**Parameters**:
- `user_input`: The original question
- `response`: The generated response to evaluate

**Note**: This metric requires embeddings in addition to LLM.


In [5]:
# Example 4: Answer Relevancy Metric

#strictness -- no of questions
scorer = AnswerRelevancy(llm=llm, embeddings=embeddings, strictness=3)

# Sample data
user_input = "What is the capital of France?"
response = "AI is the future"

# Evaluate
result = await scorer.ascore(
    user_input=user_input,
    response=response
)

print(f"Answer Relevancy Score: {result.value}")
print(f"Score Range: 0.0 to 1.0 (higher is better)")
print(f"Strictness: 3 (number of questions generated from response)")





Answer Relevancy Score: 0.03334002641347212
Score Range: 0.0 to 1.0 (higher is better)
Strictness: 3 (number of questions generated from response)


## 5. Noise Sensitivity Metric

**Purpose**: Measures how often a system makes errors by providing incorrect responses when utilizing either relevant or irrelevant retrieved documents. The metric decomposes reference and response into atomic statements and uses NLI to evaluate statement faithfulness.

**Parameters**:
- `user_input`: The original question
- `response`: The generated response to evaluate
- `reference`: The reference/gold standard answer
- `retrieved_contexts`: List of retrieved context strings
- `mode`: Either "relevant" or "irrelevant" (default: "relevant")

**Note**: Lower scores are better for this metric (it measures sensitivity to noise/errors).

In [ ]:
# Example 5: Noise Sensitivity Metric (Relevant Context Mode)
scorer = NoiseSensitivity(llm=llm, mode="relevant")

# Sample data
user_input = "What is the Life Insurance Corporation of India (LIC) known for?"
response = "The Life Insurance Corporation of India (LIC) is the largest insurance company in India, known for its vast portfolio of investments. LIC contributes to the financial stability of the country."
reference = "The Life Insurance Corporation of India (LIC) is the largest insurance company in India, established in 1956 through the nationalization of the insurance industry. It is known for managing a large portfolio of investments."
retrieved_contexts = [
    "The Life Insurance Corporation of India (LIC) was established in 1956 following the nationalization of the insurance industry in India.",
    "LIC is the largest insurance company in India, with a vast network of policyholders and huge investments.",
    "As the largest institutional investor in India, LIC manages substantial funds, contributing to the financial stability of the country.",
    "The Indian economy is one of the fastest-growing major economies in the world, thanks to sectors like finance, technology, manufacturing etc."
]

# Evaluate
result = await scorer.ascore(
    user_input=user_input,
    response=response,
    reference=reference,
    retrieved_contexts=retrieved_contexts
)

print(f"Noise Sensitivity Score (Relevant Mode): {result.value}")
print(f"Score Range: 0.0 to 1.0 (lower is better)")
print(f"Measures: How often the system makes errors with relevant contexts")


In [ ]:
# Example 5b: Noise Sensitivity Metric (Irrelevant Context Mode)
scorer_irrelevant = NoiseSensitivity(llm=llm, mode="irrelevant")

# Sample data with some irrelevant contexts
user_input = "What is the capital of France?"
response = "Paris is the capital of France."
reference = "Paris is the capital of France."
retrieved_contexts = [
    "Paris is the capital of France.",  # Relevant
    "The Eiffel Tower is located in Paris.",  # Relevant
    "Tokyo is the capital of Japan.",  # Irrelevant
    "The weather in London is often rainy."  # Irrelevant
]

# Evaluate
result = await scorer_irrelevant.ascore(
    user_input=user_input,
    response=response,
    reference=reference,
    retrieved_contexts=retrieved_contexts
)

print(f"Noise Sensitivity Score (Irrelevant Mode): {result.value}")
print(f"Score Range: 0.0 to 1.0 (lower is better)")
print(f"Measures: How often the system makes errors with irrelevant contexts")


## Summary

All metrics return a `MetricResult` object with a `value` attribute containing the score.

### Key Differences:
- **Faithfulness**: Only needs `user_input`, `response`, and `retrieved_contexts`
- **Answer Relevancy**: Only needs `user_input` and `response` (requires embeddings)
- **Context Precision**: Needs `user_input`, `reference`, and `retrieved_contexts`
- **Context Recall**: Needs `user_input`, `retrieved_contexts`, and `reference`
- **Noise Sensitivity**: Needs all four: `user_input`, `response`, `reference`, and `retrieved_contexts`

### Score Interpretation:
- **Faithfulness, Answer Relevancy, Context Precision, Context Recall**: Higher is better (0.0 to 1.0)
- **Noise Sensitivity**: Lower is better (0.0 to 1.0) - measures error rate
